# MS Project Timeline Screenshot → Structured JSON → Azure AI Search

This notebook shows how to take a screenshot of a Microsoft Project phase timeline / Gantt-style chart and turn it into structured JSON suitable for Azure AI Search.

The key idea is:

1. Use **Azure AI Document Intelligence Layout** to extract text and bounding polygons.
2. Use **OpenCV** to detect visual timeline bars as contours.
3. Convert each contour's pixel `x` positions into actual dates using OCR-detected date labels on the timeline axis.
4. Match each visual bar to the closest OCR phase label by row position.
5. Emit structured JSON records.
6. Optionally upload the records to Azure AI Search.

> Important: Screenshots vary a lot. Treat this as a strong baseline notebook, not a universal parser. For production, tune the image preprocessing and matching rules against your customer’s actual MS Project screenshot style.


In [ ]:
# Cell 1 - Install dependencies
# Run once in a fresh notebook environment.
%pip install --upgrade --quiet azure-ai-documentintelligence azure-core azure-identity azure-search-documents python-dotenv opencv-python pillow matplotlib numpy python-dateutil


## 1. Imports and configuration

This block loads Python libraries and reads configuration from environment variables.

Required for Azure Document Intelligence:

- `AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT`
- `AZURE_DOCUMENT_INTELLIGENCE_KEY`

Optional for Azure AI Search upload:

- `SEARCH_ENDPOINT`
- `SEARCH_API_KEY`
- `SEARCH_INDEX_NAME`

The notebook can still run the OpenCV contour logic without Azure Search. Document Intelligence is needed for OCR and layout extraction.


In [ ]:
# Cell 2 - Imports and environment variables
from __future__ import annotations

import os
import re
import json
import math
from dataclasses import dataclass, asdict
from datetime import datetime, date, timedelta
from pathlib import Path
from typing import Any, Iterable

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dateutil import parser as date_parser
from dotenv import load_dotenv

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

# Optional Azure AI Search upload
try:
    from azure.search.documents import SearchClient
    from azure.core.credentials import AzureKeyCredential as SearchAzureKeyCredential
except Exception:
    SearchClient = None

load_dotenv()

DOC_INTEL_ENDPOINT = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT', '').rstrip('/')
DOC_INTEL_KEY = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_KEY', '')

SEARCH_ENDPOINT = os.getenv('SEARCH_ENDPOINT', '').rstrip('/')
SEARCH_API_KEY = os.getenv('SEARCH_API_KEY', '')
SEARCH_INDEX_NAME = os.getenv('SEARCH_INDEX_NAME', 'project-timeline-index')

# Replace with your screenshot path.
IMAGE_PATH = Path(os.getenv('TIMELINE_IMAGE_PATH', 'sample-ms-project-timeline.png'))

print('Image path:', IMAGE_PATH)
print('Document Intelligence configured:', bool(DOC_INTEL_ENDPOINT and DOC_INTEL_KEY))
print('Azure AI Search configured:', bool(SEARCH_ENDPOINT and SEARCH_API_KEY))


## 2. Data structures

The notebook uses small dataclasses so each step has a clean, inspectable output.

- `OcrItem`: a text fragment with a bounding box.
- `DateTick`: a date label on the timeline axis, such as `Jan`, `Feb`, `Mar` or `01/03/2026`, plus its x-coordinate.
- `TimelineBar`: a visual bar detected by OpenCV.
- `PhaseRecord`: final structured output for Azure AI Search.


In [ ]:
# Cell 3 - Data structures
@dataclass
class OcrItem:
    text: str
    x1: float
    y1: float
    x2: float
    y2: float
    cx: float
    cy: float


@dataclass
class DateTick:
    label: str
    tick_date: date
    x: float
    y: float


@dataclass
class TimelineBar:
    x1: int
    y1: int
    x2: int
    y2: int
    width: int
    height: int
    cx: float
    cy: float
    area: float


@dataclass
class PhaseRecord:
    id: str
    projectName: str
    phaseName: str
    startDate: str
    endDate: str
    durationDays: int
    sourceFile: str
    content: str
    extractionConfidence: str
    boundingBox: dict[str, int]


## 3. Optional: create a synthetic sample image

If you do not yet have a real MS Project screenshot, this creates a very simple synthetic timeline image. It is useful for validating the OpenCV and date-mapping logic locally.

For real use, set `TIMELINE_IMAGE_PATH` to your screenshot and skip this cell.


In [ ]:
# Cell 4 - Optional synthetic sample image
from PIL import ImageDraw, ImageFont


def create_synthetic_timeline(path: Path) -> None:
    width, height = 1400, 700
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)

    # Timeline axis
    left, right = 280, 1250
    top = 90
    months = ['Jan 2026', 'Feb 2026', 'Mar 2026', 'Apr 2026', 'May 2026', 'Jun 2026']
    for i, label in enumerate(months):
        x = left + i * ((right - left) / (len(months) - 1))
        draw.line((x, top + 30, x, height - 60), fill=(220, 220, 220), width=2)
        draw.text((x - 35, top), label, fill=(0, 0, 0))

    phases = [
        ('Requirements', 0.00, 1.05, 170, (79, 129, 189)),
        ('Design',       1.00, 2.25, 250, (155, 187, 89)),
        ('Build',        2.10, 4.10, 330, (192, 80, 77)),
        ('Test',         4.00, 5.00, 410, (128, 100, 162)),
        ('Go Live',      5.00, 5.00, 500, (247, 150, 70)),
    ]

    span = (right - left) / (len(months) - 1)
    for name, start_idx, end_idx, y, colour in phases:
        draw.text((30, y - 8), name, fill=(0, 0, 0))
        x1 = left + start_idx * span
        x2 = left + end_idx * span
        if x1 == x2:
            # milestone diamond
            pts = [(x1, y - 14), (x1 + 14, y), (x1, y + 14), (x1 - 14, y)]
            draw.polygon(pts, fill=colour)
        else:
            draw.rounded_rectangle((x1, y - 13, x2, y + 13), radius=8, fill=colour)

    img.save(path)

if not IMAGE_PATH.exists():
    create_synthetic_timeline(IMAGE_PATH)
    print('Created synthetic image:', IMAGE_PATH)
else:
    print('Using existing image:', IMAGE_PATH)


## 4. Run Azure Document Intelligence Layout OCR

This block uses the Document Intelligence Layout model to extract text and bounding polygons. The Azure SDK documentation describes Layout as extracting content and structure, including words, selection marks and tables. Bounding regions include polygons with coordinates relative to the page top-left.

The notebook converts each polygon into a simple rectangle `(x1, y1, x2, y2)` and centre point `(cx, cy)`.

Why this matters:

- Date labels such as `Jan 2026` are used to build the timeline x-axis.
- Phase labels such as `Design` and `Build` are matched to the visual bars by comparing y-coordinates.


In [ ]:
# Cell 5 - Document Intelligence OCR / Layout extraction

def polygon_to_box(polygon: list[float]) -> tuple[float, float, float, float]:
    """Convert Document Intelligence polygon list to x1, y1, x2, y2."""
    xs = polygon[0::2]
    ys = polygon[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def analyse_layout_with_document_intelligence(image_path: Path) -> list[OcrItem]:
    if not DOC_INTEL_ENDPOINT or not DOC_INTEL_KEY:
        raise RuntimeError('Set AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT and AZURE_DOCUMENT_INTELLIGENCE_KEY to run OCR.')

    client = DocumentIntelligenceClient(
        endpoint=DOC_INTEL_ENDPOINT,
        credential=AzureKeyCredential(DOC_INTEL_KEY),
    )

    with image_path.open('rb') as handle:
        poller = client.begin_analyze_document(
            'prebuilt-layout',
            AnalyzeDocumentRequest(bytes_source=handle.read()),
        )
    result = poller.result()

    items: list[OcrItem] = []
    for page in result.pages or []:
        for line in page.lines or []:
            if not line.content or not line.polygon:
                continue
            x1, y1, x2, y2 = polygon_to_box(list(line.polygon))
            items.append(OcrItem(
                text=line.content.strip(),
                x1=x1,
                y1=y1,
                x2=x2,
                y2=y2,
                cx=(x1 + x2) / 2,
                cy=(y1 + y2) / 2,
            ))
    return items

# Run OCR if configured. Otherwise use synthetic OCR coordinates for the generated sample image.
if DOC_INTEL_ENDPOINT and DOC_INTEL_KEY:
    ocr_items = analyse_layout_with_document_intelligence(IMAGE_PATH)
else:
    # Approximate synthetic fallback coordinates for the generated sample only.
    ocr_items = [
        OcrItem('Jan 2026', 245, 88, 330, 112, 287, 100),
        OcrItem('Feb 2026', 438, 88, 523, 112, 480, 100),
        OcrItem('Mar 2026', 632, 88, 717, 112, 674, 100),
        OcrItem('Apr 2026', 826, 88, 911, 112, 868, 100),
        OcrItem('May 2026', 1020, 88, 1112, 112, 1066, 100),
        OcrItem('Jun 2026', 1212, 88, 1300, 112, 1256, 100),
        OcrItem('Requirements', 30, 160, 160, 185, 95, 172),
        OcrItem('Design', 30, 240, 100, 265, 65, 252),
        OcrItem('Build', 30, 320, 90, 345, 60, 332),
        OcrItem('Test', 30, 400, 80, 425, 55, 412),
        OcrItem('Go Live', 30, 490, 110, 515, 70, 502),
    ]

print('OCR items:', len(ocr_items))
for item in ocr_items[:15]:
    print(item)


## 5. Parse timeline date labels

This block identifies OCR fragments that look like timeline date labels.

Typical labels:

- `Jan 2026`
- `Feb`
- `01/03/2026`
- `2026-03-01`

The important output is a list of `DateTick` values. Each tick has:

- the parsed real date;
- the x-coordinate on the image.

Later, we map a bar's `x1` and `x2` positions onto this date axis.


In [ ]:
# Cell 6 - Parse date tick labels from OCR
MONTH_PATTERN = re.compile(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*', re.IGNORECASE)
YEAR_PATTERN = re.compile(r'(20\d{2}|19\d{2})')


def parse_date_label(text: str, default_year: int | None = None) -> date | None:
    text = text.strip()
    try:
        parsed = date_parser.parse(text, fuzzy=True, default=datetime(default_year or 2026, 1, 1))
        return parsed.date().replace(day=1)
    except Exception:
        return None


def extract_date_ticks(ocr_items: list[OcrItem], default_year: int | None = None) -> list[DateTick]:
    ticks: list[DateTick] = []
    for item in ocr_items:
        looks_like_month = bool(MONTH_PATTERN.search(item.text))
        looks_like_year = bool(YEAR_PATTERN.search(item.text))
        looks_like_numeric_date = bool(re.search(r'\d{1,2}[/-]\d{1,2}', item.text))
        if not (looks_like_month or looks_like_year or looks_like_numeric_date):
            continue
        parsed = parse_date_label(item.text, default_year=default_year)
        if parsed:
            ticks.append(DateTick(label=item.text, tick_date=parsed, x=item.cx, y=item.cy))
    ticks = sorted(ticks, key=lambda t: t.x)
    return ticks

# You can set default_year if the screenshot only says Jan, Feb, Mar.
date_ticks = extract_date_ticks(ocr_items, default_year=2026)
print('Detected date ticks:')
for tick in date_ticks:
    print(tick)


## 6. OpenCV preprocessing and contour detection

This is the key computer vision section.

The goal is to detect the coloured horizontal bars in the MS Project timeline.

The processing steps are:

1. Load image with OpenCV.
2. Convert to HSV colour space.
3. Create a mask for saturated coloured objects. This filters out black text and grey gridlines.
4. Use morphological closing to join small gaps in bars.
5. Find contours on the binary mask.
6. Convert each contour into a bounding rectangle.
7. Filter contour boxes so we keep timeline bars rather than small noise.

OpenCV contour detection works best on binary images where the object is white and the background is black. Contours are lists of boundary points; `cv2.boundingRect` converts a contour to `(x, y, width, height)`.


In [ ]:
# Cell 7 - Computer vision: detect timeline bars as contours

def load_image_bgr(path: Path) -> np.ndarray:
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f'Could not read image: {path}')
    return img


def build_bar_mask(image_bgr: np.ndarray) -> np.ndarray:
    """Build a binary mask for likely coloured timeline bars.

    This targets saturated coloured bars and ignores most black OCR text and grey gridlines.
    You will tune HSV thresholds for each customer's screenshot style.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # Saturation threshold keeps coloured bars.
    # Value threshold avoids very dark artefacts.
    lower = np.array([0, 35, 40])
    upper = np.array([179, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)

    # Remove tiny specks and close small gaps inside bars.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask


def detect_timeline_bars(image_bgr: np.ndarray, mask: np.ndarray) -> list[TimelineBar]:
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bars: list[TimelineBar] = []
    image_height, image_width = image_bgr.shape[:2]

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = cv2.contourArea(contour)

        # Filtering rules:
        # - Timeline task bars are usually wider than they are tall.
        # - Milestones can be more square, so we allow medium square objects too.
        # - Ignore very small objects, labels and noise.
        aspect_ratio = w / max(h, 1)
        looks_like_bar = w > 40 and 8 <= h <= 60 and aspect_ratio >= 2.0
        looks_like_milestone = 12 <= w <= 45 and 12 <= h <= 45 and 0.6 <= aspect_ratio <= 1.6

        if not (looks_like_bar or looks_like_milestone):
            continue
        if area < 80:
            continue
        if y < image_height * 0.12:
            # Usually the timeline axis/header, not a task bar.
            continue

        bars.append(TimelineBar(
            x1=x,
            y1=y,
            x2=x + w,
            y2=y + h,
            width=w,
            height=h,
            cx=x + w / 2,
            cy=y + h / 2,
            area=float(area),
        ))

    return sorted(bars, key=lambda b: (b.y1, b.x1))

image_bgr = load_image_bgr(IMAGE_PATH)
bar_mask = build_bar_mask(image_bgr)
bars = detect_timeline_bars(image_bgr, bar_mask)

print('Detected bars / milestones:', len(bars))
for bar in bars:
    print(bar)

plt.figure(figsize=(14, 5))
plt.imshow(bar_mask, cmap='gray')
plt.title('Binary mask used for contour detection')
plt.axis('off')
plt.show()


## 7. Visualise contours

This block overlays detected bars on the source image.

Use this as the debugging cell. If detections are poor, tune:

- HSV saturation/value thresholds in `build_bar_mask()`;
- morphology kernel size;
- width/height/aspect ratio filters in `detect_timeline_bars()`.


In [ ]:
# Cell 8 - Visualise detected contours / bars

def draw_bars(image_bgr: np.ndarray, bars: list[TimelineBar]) -> np.ndarray:
    output = image_bgr.copy()
    for i, bar in enumerate(bars, start=1):
        cv2.rectangle(output, (bar.x1, bar.y1), (bar.x2, bar.y2), (0, 255, 0), 2)
        cv2.putText(output, str(i), (bar.x1, bar.y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 128, 0), 2)
    return output

annotated = draw_bars(image_bgr, bars)
plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('Detected timeline bars and milestones')
plt.axis('off')
plt.show()


## 8. Convert contour x-coordinates into dates

This is the most important logic.

We have two pieces of information:

1. OCR date ticks: each has a real date and an image x-coordinate.
2. A detected bar: each has `x1` and `x2` pixel positions.

We convert a pixel x-position to a date using linear interpolation between the two nearest date ticks.

Example:

```text
Jan 2026 at x = 280
Feb 2026 at x = 480
Detected bar starts at x = 380
```

The bar starts halfway between January and February, so its inferred date is approximately mid-January.

Formula:

```text
ratio = (bar_x - left_tick_x) / (right_tick_x - left_tick_x)
date = left_tick_date + ratio * (right_tick_date - left_tick_date)
```

For bar end dates, use `x2`. For milestone diamonds, `x1` and `x2` are close, so start and end can be the same date.


In [ ]:
# Cell 9 - Map x-coordinate to date by interpolation

def interpolate_date_from_x(x: float, ticks: list[DateTick]) -> date:
    if len(ticks) < 2:
        raise ValueError('Need at least two date ticks to map x positions to dates.')
    ticks = sorted(ticks, key=lambda t: t.x)

    # Clamp left/right so bars outside the detected axis still get a best-effort date.
    if x <= ticks[0].x:
        return ticks[0].tick_date
    if x >= ticks[-1].x:
        return ticks[-1].tick_date

    for left, right in zip(ticks[:-1], ticks[1:]):
        if left.x <= x <= right.x:
            span_px = right.x - left.x
            ratio = (x - left.x) / span_px if span_px else 0
            span_days = (right.tick_date - left.tick_date).days
            return left.tick_date + timedelta(days=round(ratio * span_days))

    return ticks[-1].tick_date

# Show how every detected bar maps to dates.
for bar in bars:
    start = interpolate_date_from_x(bar.x1, date_ticks)
    end = interpolate_date_from_x(bar.x2, date_ticks)
    print((bar.x1, bar.x2), '=>', start.isoformat(), 'to', end.isoformat())


## 9. Match bars to phase labels

Timeline bars and phase names are usually aligned horizontally by row.

This block:

1. Filters OCR items that are not date labels.
2. Treats remaining left-side text as candidate phase labels.
3. For each bar, finds the nearest label by vertical centre `cy`.
4. Uses a maximum y-distance threshold so unrelated text is not accidentally matched.

This is a heuristic. In production, you should tune it for your customer’s timeline format.


In [ ]:
# Cell 10 - Match detected bars to OCR phase labels

def is_date_like(text: str) -> bool:
    return bool(MONTH_PATTERN.search(text) or YEAR_PATTERN.search(text) or re.search(r'\d{1,2}[/-]\d{1,2}', text))


def get_phase_label_candidates(ocr_items: list[OcrItem], image_width: int) -> list[OcrItem]:
    candidates = []
    for item in ocr_items:
        if is_date_like(item.text):
            continue
        if len(item.text.strip()) < 2:
            continue
        # Many MS Project screenshots place task names on the left.
        if item.cx < image_width * 0.45:
            candidates.append(item)
    return candidates


def match_bar_to_label(bar: TimelineBar, candidates: list[OcrItem], max_y_distance: float = 45) -> OcrItem | None:
    if not candidates:
        return None
    ranked = sorted(candidates, key=lambda item: abs(item.cy - bar.cy))
    best = ranked[0]
    if abs(best.cy - bar.cy) <= max_y_distance:
        return best
    return None

phase_candidates = get_phase_label_candidates(ocr_items, image_bgr.shape[1])
print('Phase label candidates:')
for c in phase_candidates:
    print(c.text, 'cy=', round(c.cy, 1))

print('
Bar to phase matches:')
for bar in bars:
    label = match_bar_to_label(bar, phase_candidates)
    print(bar, '=>', label.text if label else None)


## 10. Build output JSON records

This block combines:

- the matched phase label;
- the interpolated start/end dates;
- the duration;
- the source file;
- a human-readable `content` string for keyword/semantic/vector search;
- a bounding box for debugging and traceability.

This JSON can be pushed into Azure AI Search.


In [ ]:
# Cell 11 - Create structured JSON records

def build_phase_records(
    project_name: str,
    image_path: Path,
    bars: list[TimelineBar],
    ocr_items: list[OcrItem],
    date_ticks: list[DateTick],
) -> list[PhaseRecord]:
    label_candidates = get_phase_label_candidates(ocr_items, image_bgr.shape[1])
    records: list[PhaseRecord] = []

    for i, bar in enumerate(bars, start=1):
        label = match_bar_to_label(bar, label_candidates)
        phase_name = label.text if label else f'Unknown phase {i}'

        start_date = interpolate_date_from_x(bar.x1, date_ticks)
        end_date = interpolate_date_from_x(bar.x2, date_ticks)
        if end_date < start_date:
            start_date, end_date = end_date, start_date
        duration_days = max((end_date - start_date).days, 0)

        content = (
            f'Project {project_name}. '
            f'Phase {phase_name} starts on {start_date.isoformat()} and ends on {end_date.isoformat()}. '
            f'Duration is {duration_days} days. '
            f'Source file is {image_path.name}.'
        )

        records.append(PhaseRecord(
            id=f'{image_path.stem}-phase-{i:03d}',
            projectName=project_name,
            phaseName=phase_name,
            startDate=start_date.isoformat(),
            endDate=end_date.isoformat(),
            durationDays=duration_days,
            sourceFile=image_path.name,
            content=content,
            extractionConfidence='heuristic',
            boundingBox={'x1': bar.x1, 'y1': bar.y1, 'x2': bar.x2, 'y2': bar.y2},
        ))

    return records

project_name = os.getenv('PROJECT_NAME', 'MS Project Timeline')
phase_records = build_phase_records(project_name, IMAGE_PATH, bars, ocr_items, date_ticks)
output_json = [asdict(r) for r in phase_records]

print(json.dumps(output_json, indent=2))


## 11. Save JSON output

This saves the extracted phase records to a local `.json` file.


In [ ]:
# Cell 12 - Save output JSON
OUTPUT_JSON_PATH = Path(f'{IMAGE_PATH.stem}-timeline-records.json')
OUTPUT_JSON_PATH.write_text(json.dumps(output_json, indent=2), encoding='utf-8')
print('Saved:', OUTPUT_JSON_PATH.resolve())


## 12. Optional: upload records to Azure AI Search

Your Azure AI Search index should include fields similar to:

```json
id: Edm.String key
projectName: Edm.String searchable/filterable/facetable
phaseName: Edm.String searchable/filterable/facetable
startDate: Edm.DateTimeOffset filterable/sortable/facetable
endDate: Edm.DateTimeOffset filterable/sortable/facetable
durationDays: Edm.Int32 filterable/sortable
sourceFile: Edm.String searchable/filterable/facetable
content: Edm.String searchable
```

If you use vector search, add an embedding field and generate vectors for `content` before upload.


In [ ]:
# Cell 13 - Optional upload to Azure AI Search

def to_search_document(record: dict[str, Any]) -> dict[str, Any]:
    doc = dict(record)
    # Azure AI Search DateTimeOffset expects an ISO timestamp, not just yyyy-mm-dd.
    doc['startDate'] = doc['startDate'] + 'T00:00:00Z'
    doc['endDate'] = doc['endDate'] + 'T00:00:00Z'
    return doc

if SEARCH_ENDPOINT and SEARCH_API_KEY and SearchClient:
    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=SEARCH_INDEX_NAME,
        credential=SearchAzureKeyCredential(SEARCH_API_KEY),
    )
    docs = [to_search_document(r) for r in output_json]
    result = search_client.merge_or_upload_documents(documents=docs)
    for item in result:
        print(item.key, item.succeeded)
else:
    print('Azure AI Search upload skipped. Set SEARCH_ENDPOINT, SEARCH_API_KEY and SEARCH_INDEX_NAME to enable upload.')


## 13. Query examples after indexing

Keyword search:

```python
client.search(search_text='Build')
```

Date filter:

```python
client.search(
    search_text='*',
    filter="startDate ge 2026-03-01T00:00:00Z and endDate le 2026-06-30T00:00:00Z"
)
```

Semantic or hybrid search works better if you also index embeddings for the `content` field.


In [ ]:
# Cell 14 - Example date-filter query, if Search is configured
if SEARCH_ENDPOINT and SEARCH_API_KEY and SearchClient:
    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=SEARCH_INDEX_NAME,
        credential=SearchAzureKeyCredential(SEARCH_API_KEY),
    )
    results = search_client.search(
        search_text='*',
        filter="startDate ge 2026-03-01T00:00:00Z",
        select=['id', 'projectName', 'phaseName', 'startDate', 'endDate', 'durationDays'],
        top=10,
    )
    for r in results:
        print(dict(r))
else:
    print('Search query skipped. Configure Azure AI Search to run this cell.')


## 14. Production hardening checklist

- Tune HSV thresholds by screenshot style.
- Use Document Intelligence for OCR/layout, not OCR alone.
- Validate date-axis parsing for month-only, week-number and exact-date axes.
- Add human review for low-confidence extraction.
- Store both structured fields and narrative `content`.
- Use exact date fields for filtering; do not rely on semantic search for exact week/month matching.
- Add `group_ids`, `classification` and `sensitivityLabel` fields for security trimming.
- Keep the original image and detected bounding boxes for auditability.
